# A1 — Corpus Exploration (fill this)
Explore your scanned corpus: page/word counts, scan quality, script and font variety.

In [6]:
import sys
import subprocess
import importlib

# Map pip package names to their actual Python module names to prevent constant reinstalling
REQUIRED_PACKAGES = {
    "pypdf": "pypdf",
    "PyPDF2": "PyPDF2",
    "ocrmypdf": "ocrmypdf",
    "certifi": "certifi"
}

for package, module_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing missing package: {package}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package,
        ])

print("Python dependencies ready!")


# Single-cell direct downloader + OCR + explorer
# Usage: open this notebook and run the single cell; it will download the specific PDF,
# OCR it (if toolchain available), and print summaries.
import os, urllib.request, urllib.parse, shutil, re, statistics
import ssl
import certifi
from pathlib import Path

# --- Configuration ---
PDF_URL = 'https://archive.org/download/ar-raheequl-makhtoom-bangla/Ar_Raheequl_Makhtoom_Bangla.pdf'
PDF_URL = 'https://archive.org/download/ar-raheequl-makhtoom-bangla/Ar_Raheequl_Makhtoom_Bangla.pdf'
ITEM = 'ar-raheequl-makhtoom-bangla'  # Used for folder naming
OUTDIR = Path('data') / 'raw' / ITEM
OUTDIR.mkdir(parents=True, exist_ok=True)

FORCE_OCR = False           # Set to True to overwrite existing .txt files
KEEP_OCR_PDF = False        # Set to True to keep the large .ocr.pdf file generated by ocrmypdf
TESS_LANG = os.environ.get('TESSERACT_LANG', 'ben+eng+ara') # Added 'eng' for mixed texts

# 1. Direct Download
filename = urllib.parse.unquote(PDF_URL.split('/')[-1])
outpath = OUTDIR / filename

if outpath.exists() and outpath.stat().st_size > 0:
    print('Skipping existing download:', filename)
else:
    partpath = outpath.with_suffix(outpath.suffix + '.part')
    try:
        print('Downloading', filename, '...')
        
        # Load local certificates to bypass SSL verification errors
        ssl_context = ssl.create_default_context(cafile=certifi.where())
        
        with urllib.request.urlopen(PDF_URL, context=ssl_context) as response:
            with open(partpath, 'wb') as f:
                shutil.copyfileobj(response, f)
                
        partpath.replace(outpath) # Atomically finalize download
        print('Download complete!')
    except Exception as e:
        print('Failed to download:', e)
        if partpath.exists():
            partpath.unlink()

# 2. OCR Pipeline
all_files = sorted([p for p in OUTDIR.rglob('*') if p.is_file()])
pdfs = [p for p in all_files if p.suffix.lower() == '.pdf']

if not pdfs:
    print('No PDFs to OCR')
else:
    print('Found', len(pdfs), 'PDFs')
    ocrmypdf = shutil.which('ocrmypdf')
    pdftoppm = shutil.which('pdftoppm')
    tesseract = shutil.which('tesseract')

    if tesseract is None:
        TESSERACT_EXE = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

    tesseract = TESSERACT_EXE

    if tesseract is None and not os.path.exists(TESSERACT_EXE):
        raise FileNotFoundError("Tesseract OCR executable not found. Please install Tesseract or set the correct path.")
    if not os.path.exists(TESSERACT_EXE):
        raise FileNotFoundError(f"Tesseract not found: {TESSERACT_EXE}")

    tesseract = TESSERACT_EXE
    pdftotext = shutil.which('pdftotext')
    
    for pdf in pdfs:
        txtp = pdf.with_suffix('.txt')
        if txtp.exists() and not FORCE_OCR:
            print('TXT exists, skipping OCR for', pdf.name)
            continue
            
        print('OCRing', pdf.name)
        done = False
        
        # Method A: OCRmyPDF
        if ocrmypdf:
            try:
                tmp = pdf.with_suffix('.ocr.pdf')
                subprocess.check_call([
                  ocrmypdf,
                    '--skip-text',
                    '--language', TESS_LANG,
                    str(pdf),
                    str(tmp)
                ])
                if pdftotext:
                    subprocess.check_call([pdftotext, str(tmp), str(txtp)])
                    done = True
                else:
                    # Fallback to PyPDF if pdftotext isn't installed
                    try:
                        from pypdf import PdfReader
                        r = PdfReader(str(tmp))
                        text = '\n'.join(p.extract_text() or '' for p in r.pages)
                        txtp.write_text(text, encoding='utf-8')
                        done = True
                    except Exception:
                        pass
                
                # Cleanup the large intermediate OCR PDF unless requested
                if not KEEP_OCR_PDF and tmp.exists():
                    tmp.unlink()
                    
            except Exception as e:
                print('ocrmypdf failed:', e)
                
        # Method B: pdftoppm + Tesseract
        if not done and pdftoppm and tesseract:
            workdir = pdf.parent / (pdf.stem + '_pages')
            try:
                workdir.mkdir(parents=True, exist_ok=True)
                pref = str(workdir / 'page')
                subprocess.check_call([pdftoppm, '-png', str(pdf), pref])
                imgs = sorted(workdir.glob('*.png'))
                pieces = []
                for i, img in enumerate(imgs):
                    outbase = str(workdir / f'page_{i}')
                    subprocess.check_call([tesseract, str(img), outbase, '-l', TESS_LANG])
                    tfile = Path(outbase + '.txt')
                    if tfile.exists():
                        pieces.append(tfile.read_text(encoding='utf-8', errors='ignore'))
                txtp.write_text('\n\n'.join(pieces), encoding='utf-8')
                done = True
            except Exception as e:
                print('pdftoppm+tesseract failed:', e)
            finally:
                # Mandatory Cleanup of huge temporary image directories
                if workdir.exists():
                    shutil.rmtree(workdir, ignore_errors=True)
                    
        if not done:
            print('No OCR toolchain available or OCR failed for', pdf.name)

# 3. PDF Page Counts
def get_pdf_pages(path):
    try:
        from pypdf import PdfReader
        r = PdfReader(str(path))
        return len(r.pages)
    except Exception:
        pass
    try:
        import PyPDF2
        r = PyPDF2.PdfReader(str(path))
        return len(r.pages)
    except Exception:
        pass
    try:
        out = subprocess.check_output(['pdfinfo', str(path)], stderr=subprocess.DEVNULL).decode('utf-8', 'ignore')
        for line in out.splitlines():
            if line.lower().startswith('pages:'):
                return int(line.split(':', 1)[1].strip())
    except Exception:
        pass
    return None

pages = []
for p in pdfs:
    n = get_pdf_pages(p)
    print(p.name, 'pages ->', n)
    if n:
        pages.append(n)
if pages:
    print('PDF pages stats min/median/max:', min(pages), statistics.median(pages), max(pages))

# 4. Word Counts
txts = sorted([p for p in OUTDIR.rglob('*.txt') if p.is_file()])
if not txts:
    print('No OCR text files found.')
else:
    wcounts = []
    for t in txts:
        try:
            text = t.read_text(encoding='utf-8', errors='ignore')
            words = re.findall(r'\w+', text)
            wcounts.append(len(words))
            print(t.name, 'words =', len(words))
        except Exception as e:
            print('Failed reading', t, e)
    if wcounts:
        print('Text files word stats min/median/max:', min(wcounts), statistics.median(wcounts), max(wcounts))

# 5. Final Corpus Summary
print('\n--- FINAL CORPUS SUMMARY ---')
final_files = sorted([p for p in OUTDIR.rglob('*') if p.is_file()])
print('Files on disk:', len(final_files))
total_bytes = sum(p.stat().st_size for p in final_files)
print('Total size (MB):', round(total_bytes / 1024 / 1024, 2))

extcnt = {}
for p in final_files:
    ext = p.suffix.lower().lstrip('.') or 'noext'
    extcnt[ext] = extcnt.get(ext, 0) + 1
top = sorted(extcnt.items(), key=lambda x: x[1], reverse=True)[:12]

print('Top types:')
for k, v in top:
    print(' ', k, v)
print('\nSample files:')
for p in final_files[:20]:
    print(' ', p.resolve())

Python dependencies ready!
Skipping existing download: Ar_Raheequl_Makhtoom_Bangla.pdf
Found 1 PDFs
OCRing Ar_Raheequl_Makhtoom_Bangla.pdf
ocrmypdf failed: Command '['C:\\Users\\tasni\\Downloads\\Scripts\\ocrmypdf.EXE', '--skip-text', '--language', 'ben+eng+ara', 'data\\raw\\ar-raheequl-makhtoom-bangla\\Ar_Raheequl_Makhtoom_Bangla.pdf', 'data\\raw\\ar-raheequl-makhtoom-bangla\\Ar_Raheequl_Makhtoom_Bangla.ocr.pdf']' returned non-zero exit status 1.
Ar_Raheequl_Makhtoom_Bangla.pdf pages -> 558
PDF pages stats min/median/max: 558 558 558
Ar_Raheequl_Makhtoom_Bangla.txt words = 566910
Text files word stats min/median/max: 566910 566910 566910

--- FINAL CORPUS SUMMARY ---
Files on disk: 2
Total size (MB): 30.05
Top types:
  pdf 1
  txt 1

Sample files:
  C:\Users\tasni\OneDrive\Desktop\doc-agent-13\notebooks\data\raw\ar-raheequl-makhtoom-bangla\Ar_Raheequl_Makhtoom_Bangla.pdf
  C:\Users\tasni\OneDrive\Desktop\doc-agent-13\notebooks\data\raw\ar-raheequl-makhtoom-bangla\Ar_Raheequl_Makhtoom_